# Gradient Descent vs Derivative-Free Optimization

This notebook implements and compares two families of optimization methods on a non-convex 2D test function:

1. **Gradient Descent (GD)** — a first-order gradient-based method.
2. **Derivative-Free Optimization (DFO)** — a random-search style method that does not require gradient information.

The test function is multimodal and contains both a smooth convex bowl and superimposed cosine ripples, which makes it a useful benchmark for studying convergence behavior, step-size sensitivity, and the trade-offs between the two approaches.

The function used is:

$$
f(\mathbf{x}) = x_1^2 + 2x_2^2 - 0.3\cos(3\pi x_1) - 0.4\cos(4\pi x_2) + 0.7
$$

with global minimum near the origin.

## 1. Setup and the objective function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def f(x):
    """Non-convex 2D test function."""
    return (
        x[:, 0] ** 2
        + 2 * x[:, 1] ** 2
        - 0.3 * np.cos(3.0 * np.pi * x[:, 0])
        - 0.4 * np.cos(4.0 * np.pi * x[:, 1])
        + 0.7
    )

In [ ]:
def calculate_f(x1, x2):
    """Evaluate f over a grid for visualization."""
    f_x = []
    for i in range(len(x1)):
        for j in range(len(x2)):
            f_x.append(f(np.asarray([[x1[i], x2[j]]])))
    return np.asarray(f_x).reshape(len(x1), len(x2))


x1 = np.linspace(-100.0, 100.0, 400)
x2 = np.linspace(-100.0, 100.0, 400)
f_x = calculate_f(x1, x2).reshape(len(x1), len(x2))

In [ ]:
plt.figure(figsize=(6, 5))
plt.contourf(x1, x2, f_x, 100, cmap='hot')
plt.colorbar(label='f(x)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Objective function (contour plot)')
plt.show()

## 2. Gradient Descent

Gradient descent updates the current solution in the direction opposite to the gradient:

$$\mathbf{x}_{t+1} = \mathbf{x}_t - \eta \nabla f(\mathbf{x}_t)$$

For our objective the analytical gradient is:

$$
\nabla_{x_1} f(\mathbf{x}) = 2x_1 + 0.9\pi\sin(3\pi x_1), \qquad
\nabla_{x_2} f(\mathbf{x}) = 4x_2 + 1.6\pi\sin(4\pi x_2)
$$

In [ ]:
def grad(x):
    """Analytical gradient of f."""
    grad1 = 2 * x[:, 0] + 0.9 * np.pi * np.sin(3 * np.pi * x[:, 0])
    grad2 = 4 * x[:, 1] + 1.6 * np.pi * np.sin(4 * np.pi * x[:, 1])
    return np.column_stack([grad1, grad2])


class GradientDescent:
    """Vanilla gradient-descent optimizer."""

    def __init__(self, grad, step_size=0.1):
        self.grad = grad
        self.step_size = step_size

    def step(self, x_old):
        return x_old - self.step_size * self.grad(x_old)

In [ ]:
def plot_optimization_process(ax, optimizer, title, num_epochs=20, x_init=None):
    """Run an optimizer for `num_epochs` steps and plot the trajectory."""
    ax.contourf(x1, x2, f_x, 100, cmap='hot')

    x = np.asarray([[90.0, -90.0]]) if x_init is None else x_init.copy()
    x_opt = x
    for _ in range(num_epochs):
        x = optimizer.step(x)
        x_opt = np.concatenate((x_opt, x), 0)

    ax.plot(x_opt[:, 0], x_opt[:, 1], linewidth=3.0)
    ax.set_title(title)

In [ ]:
# Sweep over step sizes to study sensitivity
num_epochs = 20
step_sizes = [0.01, 0.05, 0.1, 0.25, 0.4, 0.5]

fig, axs = plt.subplots(1, len(step_sizes), figsize=(15, 2.5))
fig.suptitle('Gradient Descent — trajectories for different step sizes', y=1.05)
fig.tight_layout()

for i, step_size in enumerate(step_sizes):
    gd = GradientDescent(grad, step_size=step_size)
    plot_optimization_process(axs[i], optimizer=gd, title=f'step={step_size}')

plt.show()

### Observations on step size

- Very small step sizes (e.g. 0.01) move slowly and fail to reach the optimum within the budget.
- Moderate step sizes (~0.05-0.1) converge cleanly toward the global minimum.
- Large step sizes (>= 0.25) overshoot, oscillate, and in some cases diverge along the steeper x2 axis.

Possible remedies:
- For too-small step sizes: use momentum or Adam, or run for more iterations.
- For too-large step sizes: use a learning-rate schedule that decays over time, or switch to a line search.

## 3. Derivative-Free Optimization

When the gradient is unavailable or expensive, derivative-free optimization (DFO) is a useful alternative. The variant implemented here is a simple **random-search** procedure: at each step a random candidate is sampled from the search space, and it is accepted only if it improves on the current solution.

This is conceptually similar to a (1+1)-evolution strategy without adaptation.

In [ ]:
class RandomSearchDFO:
    """Pure random-search DFO. Accepts a candidate only if it improves the objective."""

    def __init__(self, f, search_grid_x1, search_grid_x2, step_size=None):
        self.f = f
        self.x1 = search_grid_x1
        self.x2 = search_grid_x2
        # step_size kept for API compatibility / labeling — pure random search ignores it.
        self.step_size = step_size

    def _sample(self):
        cur_x1 = np.random.choice(self.x1)
        cur_x2 = np.random.choice(self.x2)
        return np.array([[cur_x1, cur_x2]])

    def step(self, x_old):
        x_cand = self._sample()
        if self.f(x_cand) < self.f(x_old):
            return x_cand
        return x_old

In [ ]:
# Run DFO multiple times to visualize variance across runs
num_epochs = 20
runs = [0.01, 0.05, 0.1, 0.25, 0.4, 0.5]  # labels only

fig, axs = plt.subplots(1, len(runs), figsize=(15, 2.5))
fig.suptitle('Random-search DFO — trajectories across repeated runs', y=1.05)
fig.tight_layout()

for i, label in enumerate(runs):
    dfo = RandomSearchDFO(f, x1, x2, step_size=label)
    plot_optimization_process(axs[i], optimizer=dfo, title=f'run {i+1}')

plt.show()

### Observations on the DFO method

**Strengths**
- No derivatives required — works on black-box objectives.
- Trivial to implement and parallelize.
- Less sensitive to local geometry; cannot get stuck in a local minimum the way GD can.

**Weaknesses**
- Convergence is slow because samples are drawn uniformly from the entire search space.
- The number of samples needed grows quickly with dimensionality (curse of dimensionality).
- No directional information is used, so the search is uninformative.

## 4. Comparison: GD vs DFO

| Aspect              | Gradient Descent                                  | Random-search DFO                                |
|---------------------|----------------------------------------------------|--------------------------------------------------|
| Information used    | Function value + gradient                         | Function value only                              |
| Convergence speed   | Fast when gradient is well-behaved                | Slow, especially in high dimensions              |
| Robustness to local minima | Low — can get stuck                         | Higher — samples globally                        |
| Implementation cost | Need analytical or autodiff gradient              | Only need to evaluate the objective              |
| Hyperparameters     | Step size (very sensitive)                        | Sampling distribution / search bounds            |

**When to use which?**
- Use **GD** (or one of its variants like Adam) whenever you can compute or autodiff the gradient and the objective is reasonably smooth.
- Use **DFO** when the objective is non-differentiable, noisy, discrete, or only available as a black-box simulator. More sophisticated DFO methods (CMA-ES, Bayesian optimization, Nelder–Mead) close much of the gap to GD by being smarter about where to sample.